# Optimal Sampling Strateges - Multiple birth pulses and vraible sampling size

In this notebook we use synthetic viral read count data from a fully-parameterised toy population to theoretically assess determine optimal sampling study strategies, when the rodent population are assumed to follow the dynamics of the SIR algorithm with contant birth term rate with multiple birth pulses. If the estimates of the population model parameters are close to the true model parameter values that produced the toy population in the first place imply the validity of inferential approach, and therefore lend credibility when the same pipeline is used with real metaviromic data, as done in _James Hay et al. (2021)[1]_.

Similar to field studies, random samples of rodents are drawn from the simulated toy population at predifined sampling times, which satisfy the following:
 - total number of rodents sampled at each time point depends on total population size, using a constant capturing rate, but is capped at a prescribed value;
 - the sampled individuals can be either susceptible (S), infected (I) or recovered (R), with no predefined quantities of each;
 - all individuals sampled are born and alive at the time of sampling.

For each of the sampled individuals, we use the SIR model's embedded `viral_read_model` to produce viral read count data, similar to what data is produced from the field studies (byproduct in our analyses, ground truth in real studies).

Two parameter inference approaches are evaluated:
 - (1) an optimisation approach, using the CMA-ES method from *Pints [2]*, and 
 - (2) a sampling approach, using the HaarioBardenetACMC method from *Pints [2]*.

We replicate these analyses for a range of sample sizes and frequencies of sampling values, to compare the quality of parameter estimation and proportion of infected population across different sampling protocols.

**************
### References
[1] James A. Hay et al., _Estimating epidemiologic dynamics from cross-sectional viral load distributions_. Science373,**eabh0635(2021)**. DOI:10.1126/science.abh0635

[2] Clerx, M., Robinson, M., Lambert, B., Lei, C. L., Ghosh, S., Mirams, G. R., & Gavaghan, D. J.,
_Probabilistic Inference on Noisy Time Series (PINTS)_.
Journal of Open Research Software (2019), 7(1), 23. DOI:10.5334/jors.252

In [1]:
# Load necessary libraries
import numpy as np
import pandas as pd
from scipy.stats import multinomial, skew, gumbel_r
import math
import metavirommodel as mm
import metavirommodel.inference as mmi
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pints
from matplotlib import pyplot as plt
import pints.plot

# Choose array of colours for graphs and compartments names
colours = ['blue', 'red', 'green', 'purple', 'orange', 'black', 'gray', 'pink']
compartments = ['S', 'I', 'R']

# Set random seed
np.random.seed(100)

## Gillespie stochastic SIR algorithm with contant birth term rate

#### Define rodent population

In [2]:
# Set initial reproduction number
R_0 = 3

# Set initial population state S - I - R
N_init = 400
# S_init = int(N_init / R_0)
S_init = 350
I_init = N_init - S_init
R_init = 0
initial_population = [S_init, I_init, R_init]

# Set birth rate
theta = mm.BirthRateSeason([2*0.659, 0.0148])

# Set death rates
mu = 0.001
nu = 0

# Set transition rates
infect_period = 15
beta =  R_0 / infect_period
gamma = 1 / infect_period

# Coalesce into paramater vector
parameters = initial_population
parameters.extend([theta, mu, nu, beta, gamma])

# Instantiate algorithm
algorithm = mm.Metaviromodel()

# Select start and end times
start_time = 1
end_time = 360

times = list(range(start_time, end_time+1))

# Select number of experiments
num_experiments = 1

output_algorithm = []

S_history_algorithm = []
I_history_algorithm = []
R_history_algorithm = []

I_times_history_algorithm = []
R_times_history_algorithm = []

for _ in range(num_experiments):
    output, S_history, I_history, R_history, I_times_history, R_times_history = algorithm.simulate_fixed_times(parameters, start_time, end_time)
    output_algorithm.append(output)

    S_history_algorithm.append(S_history)
    I_history_algorithm.append(I_history)
    R_history_algorithm.append(R_history)

    I_times_history_algorithm.append(I_times_history)
    R_times_history_algorithm.append(R_times_history)

output_algorithm = np.asarray(output_algorithm)

### Plot output of Gillespie for the different compartments

In [3]:
# Trace names - represent the type of individuals for the simulation
trace_name = ['{}'.format(s) for s in compartments]

# Names of panels
panels = ['{} only'.format(s) for s in compartments] + ['Total Population']

fig = go.Figure()
fig = make_subplots(rows=int(np.ceil(len(panels)/2)), cols=2, subplot_titles=tuple('{}'.format(p) for p in panels))

# Add traces to the separate counts panels
for s, spec in enumerate(compartments):
    fig.add_trace(
        go.Scatter(
            y=np.mean(output_algorithm[:, :, s], axis=0).tolist(),
            x=times,
            mode='lines',
            name=trace_name[s],
            line_color=colours[s]
        ),
        row= int(np.floor(s / 2)) + 1,
        col= s % 2 + 1
    )

fig.add_trace(
    go.Scatter(
        y=np.mean(np.sum(output_algorithm, axis=2), axis=0).tolist(),
        x=times,
        mode='lines',
        name='Total Population',
        line_color='black'
    ),
    row= 2,
    col= 2
)

# Add axis labels
fig.update_layout(
    title='Counts of compartments over time:<br>IC = {}, θ = {}, μ = {}, v = {}, β = {:.2f}, γ = {:.2f}'.format(parameters[0:3], parameters[3], parameters[4], parameters[5], parameters[6], parameters[7]),
    width=1100, 
    height=600,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis2=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis2=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis3=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis3=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis4=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis4=dict(
        linecolor='black',
        title = 'Individuals')
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/Optimal-sampling-gillespie.pdf')
fig.show()

## Produce Viral read counts values

In [4]:
# Set parameter for the viral read counts model
t_eclipse = 3  # (0 days) Time from infection to initial viral growth
t_peak = 7  # (5 days ) Time from initial viral growth to peak viral load
t_switch = 5  # (9.38 days) Time from peak viral load to secondary waning phase
t_mod = 15  # (14 days) time from secondary waning phase until gumbel distribution reaches its min scale parameter
t_LOD = math.inf # ( inf days ) Time from infection until modal read counts value is equal to the limit of detection

sigma_obs = 0.25  # Initial scale parameter for the Gumbel distribution until a=teclipse+tpeak+tswitch
s_mod = 0.4  # 0.4 multiplicative factor applied to scale paramter for the Gumble distrbution - starting at t_eclipse + t_peak + t_switch + t_scle
v_zero = 2  # read counts value at time of infection
v_peak = 3880  # (20) Modal read counts value at peak viral load
v_switch = 480  # (33) Modal read counts value at a = teclipse + tpeak + tswitch
v_LOD = 2  # Limit of detection of read counts value

parameters_vl = [
    t_eclipse, t_peak, t_switch, t_mod, t_LOD,
    v_zero, v_peak, v_switch, v_LOD,
    s_mod, sigma_obs]

# Set read counts value for the suceptible and recovered individuals
VR_susc = 0

### Plot Viral read Model

In [5]:
time_from_infec = np.arange(1, 50)
vr_val = []

for ti in time_from_infec:
    ti_vr_val = []
    for _ in range(10000):
        ti_vr_val.append(algorithm.viral_read_model(parameters_vl, ti))
    vr_val.append(ti_vr_val)

vr_val = np.asarray(vr_val)

In [6]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=time_from_infec,
        x=np.mean(vr_val, axis=1),
        mode='lines',
        name='Mean Viral read',
        showlegend=False,
    )
)

fig.add_trace(
    go.Scatter(
        y=time_from_infec.tolist() + time_from_infec.tolist()[::-1],
        x=np.quantile(vr_val, 0.975, axis=1).tolist() + np.quantile(vr_val, 0.025, axis=1).tolist()[::-1],
        mode='lines',
        fill='toself',
        fillcolor='blue',
        line_color='blue',
        opacity=0.3,
        showlegend=False,
    )
)

# Add axis labels
fig.update_layout(
    width=500, 
    height=500,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Mean Viral read',
        autorange='reversed'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Time since infection'),
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/Optimal_sampling_Viral_read_model.pdf')
fig.show()

### Compute the history of recovered individuals that fully clear the virus and generation times distribution

#### 0 = 'not cleared'; 1 = 'cleared'

In [7]:
# Daily probability of recovered fully clearing the virus
p_addl = 0.2

R_history_clear_algorithm = []

# Go through each run experiment
for _ in range(num_experiments):
    R_history_clear = []

    # Go through each recorded day
    for t, time in enumerate(times):
        current_clear_status = []

        # If there are any recovered individual
        if len(R_times_history_algorithm[_][t]) > 0:
            # Go through each of them and
            for ind, ind_ID in enumerate(R_history_algorithm[_][t]):
                clear_status = 0

                # If they have previously cleared the virus they signal that
                if ind_ID in R_history_algorithm[_][t-1] and R_history_clear[-1][R_history_algorithm[_][t-1].index(ind_ID)] == 1:
                    clear_status = 1
                # if not, they could do it today, if their time since infection exceeds teclipse + tpeak + tswitch
                elif time > R_times_history_algorithm[_][t][ind] + t_eclipse + t_peak + t_switch:
                    clear_status = 1 - np.random.binomial(1, p = (1-p_addl)**(
                        time - R_times_history_algorithm[_][t][ind] - t_eclipse - t_peak - t_switch))

                current_clear_status.append(clear_status)

        R_history_clear.append(current_clear_status)                

    R_history_clear_algorithm.append(R_history_clear)

In [8]:
# Compute the generation times distribution, which also follows a
# right-skewed Gumbel distribution
generation_times = []

for _ in range(70):
    if _ < t_eclipse + t_peak + t_switch:
        generation_times.append(
            1-gumbel_r.cdf(
                np.log(v_LOD),
                algorithm._compute_mode_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_LOD,
                    np.log(v_zero), np.log(v_peak), np.log(v_switch), np.log(v_LOD)),
                algorithm._compute_sigma_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_mod,
                    s_mod, sigma_obs)
            ))
        
    else:
        generation_times.append(
            (1-gumbel_r.cdf(
                np.log(v_LOD),
                algorithm._compute_mode_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_LOD,
                    np.log(v_zero), np.log(v_peak), np.log(v_switch), np.log(v_LOD)),
                algorithm._compute_sigma_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_mod,
                    s_mod, sigma_obs)
            )) * (1-p_addl)**(_ - t_eclipse - t_peak - t_switch))

#### Plot generation times

In [9]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=time_from_infec,
        y=generation_times,
        mode='lines',
        name='Generation times',
        showlegend=False,
    )
)

fig.show()

## Parameter inference
In this section we test the quality of parameter inference for two distinct inference approaches: 

- (1) an optimisation approach, using the CMA-ES method from *Pints [2]*, and 
 - (2) a sampling approach, using the HaarioBardenetACMC method from *Pints [2]*,

for a range of sample sizes and frequencies of sampling values, to compare the quality of parameter estimation and proportion of infected population across different sampling protocols.

#### Sample individuals with specific frequencies and in specific batch sizes

In [10]:
freq_samplying_range = [14, 28, 42, 56]
capturing_rate_range = [0.05, 0.1, 0.2]

### 1. Optimisation method

#### Method to create Ct value data and ground truth

In [11]:
def sensitivity_analysis_run(sample_points, sample_sizes):
    vr_values = []
    vr_infec = []

    vr_susc_ids = []
    vr_infec_ids = []
    vr_recov_ids = []

    vr_time_of_recov_infec = []
    vr_time_of_infec = []
    vr_time_since_infec = []

    for _ in range(num_experiments):
        experiment_vr_values = []
        experiment_infec = []

        experiment_susc_ids = []
        experiment_infec_ids = []
        experiment_recov_ids = []

        experiment_time_of_recov_infec = []
        experiment_time_of_infec = []
        experiment_time_since_infec = []
        # At each point in time sample sample_size individuals
        for t, time in enumerate(sample_points):
            # Identify the current infections at the specified timepoint
            current_susceptibles = S_history_algorithm[_][time-1]
            current_infections = I_history_algorithm[_][time-1]
            current_recovered = R_history_algorithm[_][time-1]
            current_infection_times = I_times_history_algorithm[_][time-1]
            current_recov_infection_times = R_times_history_algorithm[_][time-1]
            current_recov_clear_virus_status = R_history_clear_algorithm[_][time-1]
            current_recov_clear_virus_status = R_history_clear_algorithm[_][time-1]

            # Sample without replacement the sample_size individuals and
            # determine their time since infection to produce Ct values
            number_selected_susc, number_selected_infec, number_selected_rec = \
                multinomial.rvs(
                    n=sample_sizes[t],
                    p=output_algorithm[_, time-1, :]/np.sum(output_algorithm[_, time-1, :])) # determine how many of those sampled are S, I and R

            # First add the Ct values for the sampled susceptibele and recovered individuals
            sampled_vr_values = [VR_susc] * number_selected_susc

            selected_individuals_susc_ids = np.random.choice(
                    current_susceptibles,
                    size=number_selected_susc,
                    replace=False).tolist() # determine the ids of those sampled Ss
            
            if len(current_recov_infection_times) > 0:
                # If we have at least one selected recovered
                selected_individuals_indices = np.random.choice(
                    range(len(current_recov_infection_times)),
                    size=number_selected_rec,
                    replace=False).tolist() # determine the indices of those sampled Rs
            
                selected_individuals_rec_ids = [current_recovered[_] for _ in selected_individuals_indices]

                # Determine the time of infection of those sampled Rs
                selected_individuals_recov_infec_times = [current_recov_infection_times[_] for _ in selected_individuals_indices]

                sample_time_since_infec = time - selected_individuals_recov_infec_times # determine how long since infection for selected Rs

                # Determine the clearence of infection of those sampled Rs
                selected_individuals_clear_virus_status = [current_recov_clear_virus_status[_] for _ in selected_individuals_indices]

                # Run viral read model to determine individual viral read counts for each sample
                for i, ti in enumerate(sample_time_since_infec):
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, ti) * selected_individuals_clear_virus_status[i])

            elif number_selected_rec > 0:
                # If initial step when no history of infection is provided
                for i in range(number_selected_rec):
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, time))
                
                sample_time_since_infec = np.zeros(number_selected_rec)
                selected_individuals_rec_ids = [] 
            else:
                sample_time_since_infec = []
                selected_individuals_rec_ids = []

            if len(current_infection_times) > 0:
                # If we have at least one selected infection
                selected_individuals_indices = np.random.choice(
                    range(len(current_infection_times)),
                    size=number_selected_infec,
                    replace=False).tolist() # determine the indices of those sampled Is
                
                # Determine the ids of those sampled Is
                selected_individuals_infec_ids = [current_infections[_] for _ in selected_individuals_indices]

                # Determine the time of infection of those sampled Is
                selected_individuals_infec_times = [current_infection_times[_] for _ in selected_individuals_indices]
            
                sample_time_since_infec = time - selected_individuals_infec_times # determine how long since infection for selected Is

                # Run Ct model to determine individual Ct counts for each sample
                for ti in sample_time_since_infec:
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, ti))
            
            elif number_selected_infec > 0:
                # If initial step when no history of infection is provided
                for i in range(number_selected_infec):
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, time))
                
                selected_individuals_infec_times = np.zeros(number_selected_infec)
                sample_time_since_infec = np.zeros(number_selected_infec)
                selected_individuals_infec_ids = [] 
            else:
                selected_individuals_infec_times = []
                sample_time_since_infec = []
                selected_individuals_infec_ids = [] 

            experiment_vr_values.append(sampled_vr_values)
            experiment_infec.append(number_selected_infec)
            
            experiment_susc_ids.append(selected_individuals_susc_ids)
            experiment_infec_ids.append(selected_individuals_infec_ids)
            experiment_recov_ids.append(selected_individuals_rec_ids)

            experiment_time_of_recov_infec.append(selected_individuals_recov_infec_times)
            experiment_time_of_infec.append(selected_individuals_infec_times)
            experiment_time_since_infec.append(sample_time_since_infec)
        
        vr_values.append(experiment_vr_values)
        vr_infec.append(experiment_infec)

        vr_susc_ids.append(experiment_susc_ids)
        vr_infec_ids.append(experiment_infec_ids)
        vr_recov_ids.append(experiment_recov_ids)

        vr_time_of_recov_infec.append(experiment_time_of_recov_infec)
        vr_time_of_infec.append(experiment_time_of_infec)
        vr_time_since_infec.append(experiment_time_since_infec)

    vr_time_of_infec_data = []

    for _ in range(num_experiments):
        experiment_vr_time_of_infec_data = pd.DataFrame(columns=['ID', 'Value'])
        for t, time in enumerate(sample_points):
            experiment_vr_time_of_infec_data = pd.concat(
                [
                    experiment_vr_time_of_infec_data,
                    pd.DataFrame({
                        'ID': vr_susc_ids[_][t] + vr_recov_ids[_][t] + vr_infec_ids[_][t],
                        'Value': [400] * len(vr_susc_ids[_][t]) + vr_time_of_recov_infec[_][t] + vr_time_of_infec[_][t]
                    })
                ])

        vr_time_of_infec_data.append(experiment_vr_time_of_infec_data)

    vr_values_data = []

    for _ in range(num_experiments):
        experiment_vr_values_data = pd.DataFrame(columns=['ID', 'TimeOfSample', 'Value'])
        for t, time in enumerate(sample_points):
            experiment_vr_values_data = pd.concat(
                [
                    experiment_vr_values_data,
                    pd.DataFrame({
                        'ID': vr_susc_ids[_][t] + vr_recov_ids[_][t] + vr_infec_ids[_][t],
                        'TimeOfSample': [time] * sample_sizes[t],
                        'Value': vr_values[_][t]
                    })
                ])

        vr_values_data.append(experiment_vr_values_data)

    mvr_inference = mmi.MVRVirReadInfer(algorithm, generation_times=generation_times)

    # Read Viral read counts and Ct values data
    mvr_inference.read_viral_read_data(vr_values_data[0], parameters_vl)

    R0_found = mvr_inference.optimisation_problem_setup()[0]

    shody_recov_freq = []

    for t in range(len(vr_values[0])):
        shody_recov_freq.append(np.divide(
            (np.where(
                (np.asarray(vr_values[0][t]) > 130) & (np.asarray(vr_values[0][t]) < 150))[0]).shape[0], 
            sample_sizes))

    return vr_values_data, R0_found, shody_recov_freq, np.divide(vr_infec[0], sample_sizes)

In [12]:
# Transform birth rate and death rate of infected into function format for inference method
parameters[5] = lambda _: nu

#### Method to run inference with Viral read count data and plot inferred trajectories against ground truth

In [13]:
def routine_run(freq_samplying, capturing_rate):
    sample_points = np.arange(5, 320, freq_samplying)

    # Allow varyiable sample sizes
    sample_sizes = np.array(
        [int(np.floor(min(30, np.sum(output_algorithm[0, time-1, :]) * capturing_rate))) for time in sample_points])
    
    print(sample_sizes)

    # For each choice of sample size and frequency infer parameters: 
    vr_values_data, R0_found, shody_recov_freq, infec_freq_sample = sensitivity_analysis_run(sample_points, sample_sizes)

    mvr_inference = mmi.MVRVirReadInfer(algorithm, generation_times=generation_times)
    mvr_inference.read_viral_read_data(vr_values_data[0], parameters_vl)
    mvr_inference._create_posterior()

    parameters[4] = R0_found[1]
    parameters[6] = R0_found[0] / infect_period

    theta_found = []
    infec_found = []
    for _ in range(1000):
        output_found = algorithm.simulate_fixed_times(parameters, start_time, end_time)[0]

        theta_found.append(np.divide(output_found[:, 1], np.sum(output_found, axis=1)))
        infec_found.append(output_found[:, 1])

    theta_found = np.array(theta_found)
    infec_found = np.array(infec_found)

    theta_found_mean = np.mean(theta_found, axis=0)
    theta_found_upper = np.quantile(theta_found, 0.975, axis=0)
    theta_found_lower = np.quantile(theta_found, 0.025, axis=0)

    infec_found_mean = np.mean(infec_found, axis=0)
    infec_found_upper = np.quantile(infec_found, 0.975, axis=0)
    infec_found_lower = np.quantile(infec_found, 0.025, axis=0)

    output_found_det = mvr_inference.loglikelihood._run_sir_model(
        parameters, np.arange(max(mvr_inference.loglikelihood._vr_sampled_times))
    )
    theta_found_det = np.divide(output_found_det[:, 1], np.sum(output_found_det, axis=1))
    infec_found_det = output_found_det[:, 1]

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            y=(output_algorithm[0, :, 1] / np.mean(np.sum(output_algorithm, axis=2), axis=0)).tolist(),
            x=times,
            mode='lines',
            name='True freq',
            line_color='red'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_freq_sample.tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq',
            line_color='blue'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=(1 - np.array(shody_recov_freq)).tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq (extreme I)',
            line_color='green'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_det.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq - det',
            line_color='black',
            line_dash='dash'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_mean.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq',
            line_color='black'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=times + times[::-1],
            y=theta_found_upper.tolist() + theta_found_lower.tolist()[::-1],
            fill='toself',
            fillcolor='black',
            line_color='black',
            opacity=0.3,
            mode='lines',
            showlegend=False,
            name='Inferred freq'
        )
    )

    # Add axis labels
    fig.update_layout(
        title='Sensitivity analysis: Frequency:{} days; Capturing rate:{}'.format(freq_samplying, capturing_rate),
        width=700, 
        height=400,
        plot_bgcolor='white',
        xaxis=dict(
            linecolor='black',
            title = 'Time (days)'
            ),
        yaxis=dict(
            linecolor='black',
            title = 'Individuals'),
        )

    fig.write_image('images/Var_sample_Optimal_sampling_Viral_read_CredInt_Freq_{}_Capture_rate_{}.pdf'.format(freq_samplying, capturing_rate))
    fig.show()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            y=(output_algorithm[0, :, 1]).tolist(),
            x=times,
            mode='lines',
            name='True freq',
            line_color='red'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_found_det.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq - det',
            line_color='black',
            line_dash='dash'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_found_mean.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq',
            line_color='black'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=times + times[::-1],
            y=infec_found_upper.tolist() + infec_found_lower.tolist()[::-1],
            fill='toself',
            fillcolor='black',
            line_color='black',
            opacity=0.3,
            mode='lines',
            showlegend=False,
            name='Inferred freq'
        )
    )

    # Add axis labels
    fig.update_layout(
        title='Total Infections: Frequency:{} days; Capturing rate:{}'.format(freq_samplying, capturing_rate),
        width=700, 
        height=400,
        plot_bgcolor='white',
        xaxis=dict(
            linecolor='black',
            title = 'Time (days)'
            ),
        yaxis=dict(
            linecolor='black',
            title = 'Individuals'))

    fig.write_image('images/Var_sample_Total_Infec_SIR_Viral_read__Freq_{}_Sample_size_{}.pdf'.format(freq_samplying, capturing_rate))
    fig.show()

#### Run optimisation-based inference method for multiple sampling protcols

In [14]:
routine_run(freq_samplying_range[0], capturing_rate_range[0])

[19 20 20 21 22 24 24 24 24 24 23 23 23 22 23 23 25 25 27 27 27 27 26]
Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6


/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:141: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:157: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Iter. Eval. Best      Current   Time    
0     6     -2026.196 -2026.196   0:19.5
1     12    -1907.292 -1907.292   0:40.1
2     18    -1748.346 -1748.346   1:02.2
3     24    -1707.011 -1707.011   1:18.5
20    126   -1689.832 -1691.482   7:55.3
40    246   -1687.379 -1687.63   19:29.0
60    366   -1687.294 -1687.294  31:41.0
80    486   -1687.178 -1687.178  43:34.9
100   606   -1687.176 -1687.177  55:23.0
120   726   -1687.175 -1687.175  67:11.2
129   774   -1687.175 -1687.175  71:48.1
Halting: No significant change in best function evaluation for 100 iterations.
[2.99981224 0.01087655] -1687.1751002829164
Optimisation phase is finished.


In [15]:
routine_run(freq_samplying_range[1], capturing_rate_range[0])

[19 20 22 24 24 23 23 23 25 27 27 26]
Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6


/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:141: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:157: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Iter. Eval. Best      Current   Time    
0     6     -1097.014 -1097.014   0:18.9
1     12    -1052.202 -1052.202   0:36.3
2     18    -972.8463 -972.8463   0:53.9
3     24    -913.9577 -913.9577   1:08.8
20    126   -870.0831 -870.1348   5:08.2
40    246   -870.0831 -870.3257  10:53.0
60    366   -870.0831 -870.2934  16:46.2
80    486   -869.8946 -869.8946  22:41.4
100   606   -868.2194 -868.2322  27:33.1
120   726   -868.2177 -868.2243  32:16.0
140   846   -868.2106 -868.2106  36:58.1
160   966   -868.2105 -868.2105  41:53.3
180   1086  -868.2105 -868.2105  46:50.7
183   1098  -868.2105 -868.2105  47:18.3
Halting: No significant change in best function evaluation for 100 iterations.
[2.99591393 0.011     ] -868.2104655298185
Optimisation phase is finished.


In [16]:
routine_run(freq_samplying_range[2], capturing_rate_range[0])

[19 21 24 24 23 23 27 27]
Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6


/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:157: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Iter. Eval. Best      Current   Time    
0     6     -775.3767 -775.3767   0:13.2
1     12    -729.956  -729.956    0:25.7
2     18    -645.7889 -645.7889   0:37.9
3     24    -645.7889 -653.2817   0:46.0
20    126   -624.9848 -633.3725   3:14.9
40    246   -623.4921 -626.6707   6:29.3
60    366   -623.4921 -624.2707   9:55.8
80    486   -623.4921 -624.7112  13:36.2
100   606   -623.4921 -624.6817  17:36.2
120   726   -623.4921 -624.6409  21:46.8
123   738   -623.4921 -624.6399  22:12.1
Halting: No significant change in best function evaluation for 100 iterations.
[3.00732656 0.01098446] -623.4921472399492
Optimisation phase is finished.


In [17]:
routine_run(freq_samplying_range[3], capturing_rate_range[0])

[19 22 24 23 25 27]
Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6


/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:141: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:157: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Iter. Eval. Best      Current   Time    
0     6     -527.7859 -527.7859   0:09.0
1     12    -527.7859 -527.9951   0:17.0
2     18    -507.8588 -507.8588   0:24.9
3     24    -475.3518 -475.3518   0:32.9
20    126   -437.4905 -438.0435   2:18.5
40    246   -437.4905 -437.5022   4:13.6
60    366   -437.4905 -437.5289   6:21.6
80    486   -437.4742 -437.4782   8:14.1
100   606   -437.4646 -437.4647  10:33.2
106   636   -437.4634 -437.4638  11:10.4
Halting: No significant change in best function evaluation for 100 iterations.
[2.99876094 0.01099999] -437.4633887883356
Optimisation phase is finished.


In [18]:
routine_run(freq_samplying_range[0], capturing_rate_range[1])

[30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30]
Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6


/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:141: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:157: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Iter. Eval. Best      Current   Time    
0     6     -2566.015 -2566.015   0:41.1
1     12    -2319.757 -2319.757   1:21.2
2     18    -2319.757 -2360.78    1:54.9
3     24    -2221.718 -2221.718   2:15.1
20    126   -2221.718 -2226.487  10:00.4
40    246   -2216.019 -2216.093  21:03.0
60    366   -2215.878 -2215.879  31:34.2
80    486   -2215.872 -2215.873  41:16.8
100   606   -2215.841 -2215.841  48:23.1
120   726   -2215.155 -2215.207  55:26.5
140   846   -2215.155 -2215.16   61:50.4
160   966   -2215.126 -2215.126  67:00.3
180   1086  -2215.115 -2215.115  72:08.8
200   1206  -2215.113 -2215.113  77:18.5
213   1278  -2215.113 -2215.113  80:23.4
Halting: No significant change in best function evaluation for 100 iterations.
[3.0017823  0.01075984] -2215.11298753997
Optimisation phase is finished.


In [19]:
routine_run(freq_samplying_range[1], capturing_rate_range[1])

[30 30 30 30 30 30 30 30 30 30 30 30]
Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6


/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:141: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:157: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Iter. Eval. Best      Current   Time    
0     6     -1366.791 -1366.791   0:09.6
1     12    -1232.718 -1232.718   0:17.7
2     18    -1185.74  -1185.74    0:24.6
3     24    -1157.327 -1157.327   0:32.8
20    126   -1108.365 -1109.359   2:11.4
40    246   -1107.309 -1107.518   4:32.3
60    366   -1107.25  -1107.349   7:08.7
80    486   -1107.221 -1107.224   9:36.4
100   606   -1107.218 -1107.218  12:16.2
120   726   -1107.218 -1107.218  14:50.7
138   828   -1107.218 -1107.218  17:03.1
Halting: No significant change in best function evaluation for 100 iterations.
[2.99632281 0.01098707] -1107.2179952992087
Optimisation phase is finished.


In [20]:
routine_run(freq_samplying_range[2], capturing_rate_range[1])

[30 30 30 30 30 30 30 30]
Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6


/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:141: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:157: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Iter. Eval. Best      Current   Time    
0     6     -953.7435 -953.7435   0:07.5
1     12    -953.7435 -954.5848   0:13.1
2     18    -920.2946 -920.2946   0:18.9
3     24    -872.2272 -872.2272   0:24.5
20    126   -765.258  -770.8266   1:31.3
40    246   -765.258  -765.4766   3:00.1
60    366   -764.5291 -764.8396   4:33.7
80    486   -764.5291 -764.7749   6:19.3
100   606   -764.5291 -764.624    8:05.4
120   726   -764.5291 -764.6164   9:52.0
140   846   -764.5291 -764.6119  11:38.5
149   894   -764.5291 -764.6118  12:21.5
Halting: No significant change in best function evaluation for 100 iterations.
[2.99727156 0.01097823] -764.5290892944039
Optimisation phase is finished.


In [21]:
routine_run(freq_samplying_range[3], capturing_rate_range[1])

[30 30 30 30 30 30]
Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6


/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:141: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:157: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Iter. Eval. Best      Current   Time    
0     6     -553.085  -553.085    0:03.9
1     12    -511.6159 -511.6159   0:08.5
2     18    -511.6159 -569.4294   0:10.5
3     24    -494.6039 -494.6039   0:14.9
20    126   -488.7249 -489.1756   0:59.0
40    246   -488.5468 -488.5484   2:08.8
60    366   -488.5142 -488.5154   3:18.0
80    486   -488.3341 -488.3415   4:29.0
100   606   -488.3133 -496.4228   5:38.5
115   690   -488.3133 -488.3535   6:28.5
Halting: No significant change in best function evaluation for 100 iterations.
[3.0017864  0.01075982] -488.31331506133887
Optimisation phase is finished.


In [22]:
routine_run(freq_samplying_range[0], capturing_rate_range[2])

[30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30]
Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6


/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:141: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:157: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Iter. Eval. Best      Current   Time    
0     6     -2615.704 -2615.704   0:16.4
1     12    -2333.618 -2333.618   0:33.1
2     18    -2269.191 -2269.191   0:47.0
3     24    -2234.292 -2234.292   1:03.6
20    126   -2192.686 -2202.723   4:17.0
40    246   -2190.782 -2201.713   9:07.9
60    366   -2190.782 -2196.817  14:10.0
80    486   -2190.782 -2196.7    19:23.1
100   606   -2190.782 -2196.694  24:33.1
120   726   -2190.782 -2196.365  29:44.6
139   834   -2190.782 -2196.031  34:20.9
Halting: No significant change in best function evaluation for 100 iterations.
[3.00328854 0.01075026] -2190.7819153974137
Optimisation phase is finished.


In [23]:
routine_run(freq_samplying_range[1], capturing_rate_range[2])

[30 30 30 30 30 30 30 30 30 30 30 30]
Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6


/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:141: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:157: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Iter. Eval. Best      Current   Time    
0     6     -1245.516 -1245.516   0:09.8
1     12    -1132.059 -1132.059   0:18.3
2     18    -1132.059 -1147.76    0:22.3
3     24    -1132.059 -1134.448   0:30.6
20    126   -1078.739 -1082.757   2:06.6
40    246   -1075.721 -1075.721   4:25.9
60    366   -1075.58  -1075.58    7:01.2
80    486   -1075.548 -1075.548   9:36.9
100   606   -1075.547 -1075.547  12:11.6
120   726   -1075.547 -1075.547  15:01.2
125   750   -1075.547 -1075.547  15:33.7
Halting: No significant change in best function evaluation for 100 iterations.
[2.99632281 0.01098707] -1075.547265529385
Optimisation phase is finished.


In [24]:
routine_run(freq_samplying_range[2], capturing_rate_range[2])

[30 30 30 30 30 30 30 30]
Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6


/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:141: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:157: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Iter. Eval. Best      Current   Time    
0     6     -895.5346 -895.5346   0:06.0
1     12    -889.5761 -889.5761   0:11.9
2     18    -876.1984 -876.1984   0:17.3
3     24    -862.2659 -862.2659   0:22.6
20    126   -718.4379 -719.3303   1:34.1
40    246   -713.6865 -719.2915   3:06.4
60    366   -713.6865 -717.8874   4:28.1
80    486   -713.6865 -717.1238   5:50.7
100   606   -713.6865 -716.8638   7:13.7
120   726   -713.6865 -716.8394   8:39.1
125   750   -713.6865 -716.8375   8:59.2
Halting: No significant change in best function evaluation for 100 iterations.
[2.99632277 0.01099645] -713.6864677957599
Optimisation phase is finished.


In [25]:
routine_run(freq_samplying_range[3], capturing_rate_range[2])

[30 30 30 30 30 30]
Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6


/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:141: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86254/2542007905.py:157: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Iter. Eval. Best      Current   Time    
0     6     -652.9715 -652.9715   0:04.4
1     12    -605.2727 -605.2727   0:09.3
2     18    -604.7879 -604.7879   0:10.5
3     24    -604.7879 -648.321    0:13.8
20    126   -569.9858 -572.7425   1:09.2
40    246   -569.5063 -569.8473   2:17.0
60    366   -569.5063 -569.7405   3:20.5
80    486   -569.5063 -569.738    4:15.9
100   606   -569.5063 -569.7379   5:18.3
120   726   -569.5063 -569.6999   6:29.4
133   798   -569.5063 -569.6957   7:15.3
Halting: No significant change in best function evaluation for 100 iterations.
[2.9970906  0.01098295] -569.5062706811278
Optimisation phase is finished.
